In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import config

# Load model & labels
interpreter = tf.lite.Interpreter(model_path=str(config.OUTPUT_TFLITE))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

with open(config.OUTPUT_LABELS, 'r') as f:
    labels = [line.strip() for line in f if line.strip()]

# Open PC Webcam (CAP_DSHOW for smooth Windows capture)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print(' Live Webcam Started!')
print(" Hold up animal photos inside the bounding box. Press 'q' to stop.")

while True:
    ret, frame = cap.read()
    if not ret:
        print('Failed to grab frame.')
        break

    # 1. Crop Center Region of Interest (ROI) to avoid 16:9 distortion
    h, w = frame.shape[:2]
    min_dim = min(h, w)
    x1 = (w - min_dim) // 2
    y1 = (h - min_dim) // 2
    x2 = x1 + min_dim
    y2 = y1 + min_dim
    roi = frame[y1:y2, x1:x2]

    # 2. Preprocess ROI for MobileNetV3
    rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (config.INPUT_WIDTH, config.INPUT_HEIGHT))
    input_data = np.expand_dims(resized, axis=0).astype(np.float32)

    # 3. Inference (Forward Pass)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])[0]

    # 4. Get Top Prediction
    top_idx = int(np.argmax(output))
    pred_label = labels[top_idx]
    confidence = float(output[top_idx])

    # 5. Draw Object-Detection Style Bounding Box around target ROI
    box_color = (0, 230, 0) if confidence >= config.CONFIDENCE_THRESHOLD else (0, 140, 255)
    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 3)

    # 6. Draw Attached Label Badge on top of bounding box
    label_text = f'{pred_label} {confidence * 100:.1f}%'
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    font_thickness = 2
    (text_w, text_h), _ = cv2.getTextSize(label_text, font, font_scale, font_thickness)
    badge_y1 = max(0, y1 - text_h - 12)
    badge_y2 = badge_y1 + text_h + 12
    badge_x2 = min(w, x1 + text_w + 16)

    cv2.rectangle(frame, (x1, badge_y1), (badge_x2, badge_y2), box_color, -1)
    cv2.putText(
        frame,
        label_text,
        (x1 + 8, badge_y2 - 8),
        font,
        font_scale,
        (0, 0, 0),
        font_thickness,
        cv2.LINE_AA,
    )

    cv2.imshow("Edge AI Animal Detection (Press 'q' to Quit)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('Webcam closed.')


C:\Users\sabbu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


 Live Webcam Started!
 Hold up animal photos to your webcam. Press 'q' on the camera window to stop.
